# Phase 15 — Standalone Inference Module

This notebook designs the boundary between Data Science and Backend Engineering. 
We define the `load_model`, `preprocess`, and `predict` functions. Then, we save these functions as a standalone `.py` script (`ml/src/inference/predict.py`) so the FastAPI backend can import them cleanly.

In [1]:
import os

INFERENCE_DIR = '../src/inference'
os.makedirs(INFERENCE_DIR, exist_ok=True)

In [2]:
%%writefile ../src/inference/predict.py
import os
import json
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as transforms

class TinyCNN(nn.Module):
    def __init__(self):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 7 * 7, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

class PneumoniaPredictor:
    def __init__(self, models_dir: str):
        self.models_dir = models_dir
        self.model = None
        self.model_config = None
        self.class_mapping = None
        self.transform = None
        self.load_model()

    def load_model(self):
        # Load configs
        with open(os.path.join(self.models_dir, 'model_config.json'), 'r') as f:
            self.model_config = json.load(f)
        with open(os.path.join(self.models_dir, 'class_mapping.json'), 'r') as f:
            self.class_mapping = json.load(f)
            
        # Initialize and load weights
        self.model = TinyCNN()
        weights_path = os.path.join(self.models_dir, self.model_config['pytorch_checkpoint'])
        self.model.load_state_dict(torch.load(weights_path, map_location=torch.device('cpu')))
        self.model.eval()
        
        # Define preprocessing
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

    def preprocess(self, image: Image.Image):
        # Ensure grayscale and 28x28
        image = image.convert('L').resize((28, 28))
        tensor = self.transform(image)
        return tensor.unsqueeze(0)  # Add batch dimension (1, 1, 28, 28)

    def predict(self, image: Image.Image) -> dict:
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load_model() first.")
            
        tensor = self.preprocess(image)
        
        with torch.no_grad():
            output = self.model(tensor)
            prob = torch.sigmoid(output).item()
            
        predicted_idx = 1 if prob >= 0.5 else 0
        confidence = prob if predicted_idx == 1 else (1 - prob)
        
        return {
            "predicted_class": self.class_mapping[str(predicted_idx)],
            "confidence": round(confidence, 4),
            "model_version": self.model_config['version']
        }


Overwriting ../src/inference/predict.py


In [3]:
# ── Testing the Inference Module Locally ──────────────────────────────────────
import sys
sys.path.append('../src/inference')
from predict import PneumoniaPredictor
import numpy as np
import json
from PIL import Image

# Create predictor instance (Model loading happens ONCE here)
print("Initializing Predictor...")
predictor = PneumoniaPredictor(models_dir='../models/')

# Create a dummy 28x28 grayscale image simulating a medical scan input
dummy_img_array = np.random.randint(0, 255, (28, 28), dtype=np.uint8)
dummy_image = Image.fromarray(dummy_img_array)

# Run prediction
print("Running Prediction on sample image...")
result = predictor.predict(dummy_image)

print("\n=== Inference Result ===")
print(json.dumps(result, indent=2))

Initializing Predictor...
Running Prediction on sample image...

=== Inference Result ===
{
  "predicted_class": "Pneumonia",
  "confidence": 0.8802,
  "model_version": "1.0.0"
}
